# VirtualFit AI — FASHN VTON v1.5 Free GPU Benchmark
### Genuine AI Virtual Try-On Verification on Free Cloud Compute (Google Colab / Kaggle)

**Project Requirements & Guardrails:**
- **Budget:** Strictly ₹0 (Free tier T4 GPU only).
- **Model:** [FASHN VTON v1.5](https://github.com/fashn-AI/fashn-vton-1.5) (Official Weights: [Hugging Face](https://huggingface.co/fashn-ai/fashn-vton-1.5)).
- **Licence:** Apache-2.0 (Verified for commercial use).
- **Zero Fake Outputs:** Genuine neural diffusion inference only. No simulations, placeholders, or mocked images.

---
**Pre-flight Instruction:**
Ensure you have enabled a **Free T4 GPU** before running this notebook:
1. Go to menu: `Runtime` → `Change runtime type`.
2. Under *Hardware accelerator*, select **T4 GPU**.
3. Click **Save**.


## Step 1: Free GPU Availability & VRAM Pre-Flight Check
This cell verifies that a free NVIDIA GPU is attached before downloading any model weights or dependencies. If no GPU is detected, execution halts immediately.

In [ ]:
import sys
import torch

print("=" * 60)
print("STEP 1: HARDWARE PRE-FLIGHT VERIFICATION")
print("=" * 60)

if not torch.cuda.is_available():
    print("\n[CRITICAL HALT] No NVIDIA GPU detected!")
    print("Please enable free T4 GPU in Colab:")
    print("  -> Menu: Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save.")
    print("Stopping execution to prevent unaccelerated CPU overload.\n")
    raise SystemExit("Execution stopped: Free GPU required for benchmark.")

gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)
total_vram_gb = gpu_props.total_memory / (1024 ** 3)

print(f"[SUCCESS] GPU Detected: {gpu_name}")
print(f"[SUCCESS] Total Dedicated VRAM: {total_vram_gb:.2f} GB")
print(f"[SUCCESS] CUDA Compute Capability: {gpu_props.major}.{gpu_props.minor}")
print(f"[SUCCESS] PyTorch Version: {torch.__version__}")
print("=" * 60)

# Display nvidia-smi status
!nvidia-smi

## Step 2: Clone Official Repository & Verify Licence
Clones the official FASHN VTON v1.5 repository and programmatically validates its **Apache-2.0** commercial licence.

In [ ]:
import os

print("=" * 60)
print("STEP 2: REPOSITORY & LICENCE VERIFICATION")
print("=" * 60)

# Clone official repository if not already present
if not os.path.exists("fashn-vton-1.5"):
    !git clone https://github.com/fashn-AI/fashn-vton-1.5.git

%cd fashn-vton-1.5

# Verify Licence File
license_file = "LICENSE"
if os.path.exists(license_file):
    with open(license_file, "r") as f:
        license_text = f.read()
    if "Apache License" in license_text and "Version 2.0" in license_text:
        print("\n[LICENCE VERIFIED] Apache-2.0 detected in official repository.")
        print("[LICENCE VERIFIED] Commercial use: PERMITTED without royalty or per-generation fees.")
    else:
        print("\n[WARNING] Unexpected licence text:", license_text[:200])
else:
    print("\n[WARNING] LICENSE file not found in root.")

print("=" * 60)

## Step 3: Install Dependencies
Installs the official package dependencies directly from `pyproject.toml`.

In [ ]:
print("=" * 60)
print("STEP 3: INSTALLING MODEL DEPENDENCIES")
print("=" * 60)

# Install editable package and dependencies
!pip install -q -e .

# Verify critical imports
from PIL import Image
import onnxruntime
from fashn_vton import TryOnPipeline

print("\n[SUCCESS] Pipeline dependencies successfully loaded.")
print("ONNX Runtime version:", onnxruntime.__version__)
print("ONNX Runtime Providers:", onnxruntime.get_available_providers())
print("=" * 60)

## Step 4: Download Official Model Weights (~2.2 GB)
Downloads the genuine FASHN VTON 1.5 safetensors and human pose/parsing model weights.

In [ ]:
import time

print("=" * 60)
print("STEP 4: DOWNLOADING OFFICIAL WEIGHTS")
print("=" * 60)

weights_dir = "./weights"
os.makedirs(weights_dir, exist_ok=True)

start_dl = time.perf_counter()
!python scripts/download_weights.py --weights-dir ./weights
dl_time = time.perf_counter() - start_dl

print(f"\n[SUCCESS] Weights downloaded in {dl_time:.2f} seconds.")
!ls -lh ./weights
print("=" * 60)

## Step 5: Execute Genuine AI Virtual Try-On Benchmark
Runs genuine neural diffusion inference using the official sample customer and garment images.
Measures:
- Model initialization time
- Inference latency
- Peak GPU VRAM allocated
- Output image dimensions

In [ ]:
import time
import gc
from PIL import Image
from fashn_vton import TryOnPipeline

print("=" * 60)
print("STEP 5: GENUINE AI INFERENCE EXECUTION")
print("=" * 60)

# Ensure clean GPU state before benchmark
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Official test image paths
person_path = "examples/data/model.webp"
garment_path = "examples/data/garment.webp"

assert os.path.exists(person_path), f"Missing official test person: {person_path}"
assert os.path.exists(garment_path), f"Missing official test garment: {garment_path}"

person_img = Image.open(person_path).convert("RGB")
garment_img = Image.open(garment_path).convert("RGB")

print(f"Input Person Image  : {person_path} ({person_img.size[0]}x{person_img.size[1]})")
print(f"Input Garment Image : {garment_path} ({garment_img.size[0]}x{garment_img.size[1]})")
print(f"Category            : tops")
print("-" * 60)

# 1. Initialize Pipeline
init_start = time.perf_counter()
print("Loading FASHN VTON v1.5 pipeline into GPU...")
pipeline = TryOnPipeline(weights_dir="./weights")
init_time = time.perf_counter() - init_start
print(f"[BENCHMARK] Model Pipeline Loaded in: {init_time:.2f} seconds")

# 2. Run Inference
print("\nCommencing neural diffusion inference (30 steps)...")
infer_start = time.perf_counter()

result_img = pipeline(
    person_image=person_img,
    garment_image=garment_img,
    category="tops",
    num_timesteps=30,
)

infer_time = time.perf_counter() - infer_start

# 3. Measure Peak GPU VRAM
peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
peak_reserved_mb = torch.cuda.max_memory_reserved() / (1024 ** 2)

# 4. Save Genuine Output
output_filename = "fashn_vton_benchmark_output.png"
result_img.save(output_filename)

print("=" * 60)
print("BENCHMARK RESULTS SUMMARY:")
print("=" * 60)
print(f"- Device Utilized       : {torch.cuda.get_device_name(0)}")
print(f"- Inference Duration    : {infer_time:.2f} seconds")
print(f"- Peak VRAM Allocated   : {peak_vram_mb:.1f} MB ({peak_vram_mb / 1024:.2f} GB)")
print(f"- Peak VRAM Reserved    : {peak_reserved_mb:.1f} MB ({peak_reserved_mb / 1024:.2f} GB)")
print(f"- Output Dimensions     : {result_img.size[0]} x {result_img.size[1]} pixels")
print(f"- Output File Path      : {os.path.abspath(output_filename)}")
print(f"- Status                : 100% GENUINE AI INFERENCE COMPLETED")
print("=" * 60)

## Step 6: Visual Comparison & Output Inspection
Renders the input customer photo, target garment, and the final synthesized AI try-on image side-by-side.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 8))

axes[0].imshow(person_img)
axes[0].set_title("1. Input Customer Photo", fontsize=14, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(garment_img)
axes[1].set_title("2. Target Garment Image", fontsize=14, fontweight="bold")
axes[1].axis("off")

axes[2].imshow(result_img)
axes[2].set_title("3. Genuine FASHN VTON Output", fontsize=14, fontweight="bold", color="green")
axes[2].axis("off")

plt.suptitle(f"FASHN VTON v1.5 Benchmark Output ({infer_time:.2f}s on {torch.cuda.get_device_name(0)})", fontsize=16, y=0.98)
plt.tight_layout()
plt.show()

## Step 7: VirtualFit AI Prototype Manual Inference & Export
Use this cell to generate genuine try-on results for your VirtualFit AI sessions on free Google Colab compute.

**Instructions:**
1. In your local VirtualFit AI web app, prepare a try-on session and click **Export Session Inputs for Colab**.
2. Run this cell and upload the downloaded customer photo and target garment image.
3. Specify the normalized category (`tops`, `bottoms`, or `one-pieces`) and session ID.
4. The cell generates the genuine fitting image and automatically downloads `fashn_result_{session_id}.jpg`.
5. In VirtualFit AI, click **Import Colab Generated Result Image** to view the genuine AI try-on in the web app.

In [ ]:
import os
import gc
import time
import torch
from PIL import Image

try:
    from google.colab import files
    is_colab = True
except ImportError:
    is_colab = False

print("=" * 60)
print("VIRTUALFIT AI — MANUAL INFERENCE RUNNER")
print("=" * 60)

# Clean GPU cache before inference
gc.collect()
torch.cuda.empty_cache()

# 1. Input configuration
session_id = input("Enter Session ID (or press Enter for 'prototype'): ").strip() or "prototype"
category = input("Enter Garment Category [tops / bottoms / one-pieces] (default: tops): ").strip().lower() or "tops"
if category not in ("tops", "bottoms", "one-pieces"):
    print(f"Warning: Unknown category '{category}'. Defaulting to 'tops'.")
    category = "tops"

print(f"\n[Config] Session ID: {session_id}")
print(f"[Config] Category: {category}")

# 2. Image Acquisition
print("\n--- Step 1/3: Provide Customer Photo ---")
if is_colab:
    print("Please upload the customer photo downloaded from VirtualFit AI:")
    uploaded_cust = files.upload()
    cust_filename = list(uploaded_cust.keys())[0]
else:
    cust_filename = input("Enter path to customer photo: ").strip()

print("\n--- Step 2/3: Provide Target Garment Image ---")
if is_colab:
    print("Please upload the target garment image from VirtualFit AI:")
    uploaded_garm = files.upload()
    garm_filename = list(uploaded_garm.keys())[0]
else:
    garm_filename = input("Enter path to garment image: ").strip()

cust_img = Image.open(cust_filename).convert("RGB")
garm_img = Image.open(garm_filename).convert("RGB")

print(f"Customer image dimensions: {cust_img.size}")
print(f"Garment image dimensions : {garm_img.size}")

# 3. Execute Genuine Neural Inference
print("\n--- Step 3/3: Executing Genuine FASHN VTON Inference ---")
if 'pipeline' not in locals():
    print("Loading FASHN VTON v1.5 pipeline...")
    pipeline = TryOnPipeline(weights_dir="./weights")

start_t = time.perf_counter()
result = pipeline(
    person_image=cust_img,
    garment_image=garm_img,
    category=category,
    num_timesteps=30,
)
elapsed = time.perf_counter() - start_t
print(f"[SUCCESS] Genuine inference completed in {elapsed:.2f} seconds.")

# 4. Save and export
result_filename = f"fashn_result_{session_id}.jpg"
result.save(result_filename, "JPEG", quality=95)
print(f"Saved genuine AI result to: {os.path.abspath(result_filename)}")

if is_colab:
    print(f"Triggering automatic download of '{result_filename}'...")
    files.download(result_filename)
    print("Download initiated! Now return to VirtualFit AI and click 'Import Colab Generated Result Image'.")
